# 07 — Score live flights + alternative-flight recommender

Applies the fitted feature pipeline from `04_gold` to `api_silver_flights`, loads the
champion models from Unity Catalog **by alias**, scores both variants at each model's own
tuned threshold, and upserts into:

- `flight_delay_predictions` — one row per flight per scoring run
- `alternative_flight_recommendations` — up to 5 lower-risk alternatives per flight,
  same route, ±3 hours, at least 10 points better

This notebook is the other half of the loop the original project never closed. `05_train`
registers under a 3-level UC name with a `@champion` alias; this one resolves that alias
and never mentions a run ID or a version number.


In [0]:
import sys
sys.path.append("..")

import mlflow
from mlflow.tracking import MlflowClient
from delta.tables import DeltaTable
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import VectorSlicer
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src import config

mlflow.set_registry_uri(config.MLFLOW_REGISTRY_URI)
client = MlflowClient()
print(f"Registry: {mlflow.get_registry_uri()}")


Registry: databricks-uc


## Resolve the champions

`05_train` registers **one** champion per variant — whichever of RF or GBT won, after the
tie rule. Which name that is depends on the data, so this notebook cannot assume it. It
asks the registry which of the candidate names currently carries the `@champion` alias.

The decision threshold travels with the model as a version tag. Reading it here rather
than hardcoding a number is what makes retraining a promotion rather than a code change:
if the next run picks a different cut, this notebook follows it without being edited.


In [0]:
def load_champion(candidates, variant):
    """Return (model, threshold, name, version) for the current champion."""
    errors, found = {}, []
    for name in candidates:
        try:
            mv = client.get_model_version_by_alias(name, config.CHAMPION_ALIAS)
        except Exception as e:
            errors[name] = type(e).__name__
            continue

        threshold = mv.tags.get("decision_threshold")
        if threshold is None:
            # Fallback for a model registered before the tag existed.
            try:
                threshold = client.get_run(mv.run_id).data.tags.get("decision_threshold")
            except Exception:
                threshold = None
        if threshold is None:
            raise ValueError(
                f"{name} v{mv.version} carries no decision_threshold tag. Re-run 05_train; "
                "scoring at Spark's default 0.5 produced zero positive predictions for the "
                "pre-departure model."
            )

        found.append((name, mv, float(threshold)))

    if not found:
        raise RuntimeError(
            f"No @{config.CHAMPION_ALIAS} alias on any of {candidates} ({errors}). "
            "Run 05_train first."
        )

    # More than one name carrying @champion means a previous run's alias was never
    # cleared. 05_train removes the loser's alias now, but a registry that predates
    # that fix still has both. Take the most recently created version and say so
    # loudly rather than silently serving whichever was tried first.
    if len(found) > 1:
        print(f"  WARNING: {len(found)} models carry @{config.CHAMPION_ALIAS} for "
              f"{variant}: {[n for n, _, _ in found]}")
        print(f"  Taking the most recent. Re-run 05_train to clear stale aliases.")
    found.sort(key=lambda t: int(t[1].version), reverse=True)
    found.sort(key=lambda t: t[1].creation_timestamp, reverse=True)

    name, mv, threshold = found[0]
    model = mlflow.spark.load_model(
        f"models:/{name}@{config.CHAMPION_ALIAS}", dfs_tmpdir=config.ARTIFACT_VOLUME
    )
    print(f"{variant:<14} {name}  v{mv.version}  threshold={threshold:.2f}")
    return model, threshold, name, mv.version


pre_model, PRE_THRESHOLD, pre_name, pre_version = load_champion(
    [config.MODEL_GBT_PRE, config.MODEL_RF_PRE], "pre-departure"
)
in_model, IN_THRESHOLD, in_name, in_version = load_champion(
    [config.MODEL_GBT_IN, config.MODEL_RF_IN], "in-flight"
)


{"ts": "2026-09-14 13:43:25.339", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTBhMDI3LWZiODMtNzhlZi05NGE4LTJmMTU0NThhODJhNzokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5SgwI2fOf1QYQwNnR1gNQAVgBYAFoxoPV4f+t4QE=.", "context": {}}
{"ts": "2026-09-14 13:43:25.339", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTBhMDI3LWZiODMtNzhlZi05NGE4LTJmMTU0NThhODJhNzokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5SgwI2fOf1QYQwNnR1gNQAVgBYAFoxoPV4f+t4QE=.", "context": {}}
{"ts": "2026-09-14 13:43:25.339", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMTQ4NTY4MzM3ODEwNDQyNRABIAEyJDAxYTBhMDI3LWZiODMtNzhlZi05NGE4LTJmMTU0NThhODJhNzokMzNlOWQyMzctNjc2OC0zZjY3LTllMzQtZDJkYzcyZTkwOTI5

pre-departure  workspace.flights.rf_pre_departure  v8  threshold=0.16


in-flight      workspace.flights.rf_in_flight  v6  threshold=0.39


## Feature space — from the manifest, not from a magic number

The pre-departure model is trained without `dep_delay`, so scoring has to remove the same
slot. This used to be `[i for i in range(n) if i != 11]`, which is the defect `05_train`
had removed and this notebook had quietly kept: reorder `numerical_cols` in `04_gold` and
the pre-departure model starts scoring *with* departure delay, which looks like a
suspiciously good prediction rather than an error.

The manifest `04_gold` writes is the same source `05_train` reads, so both sides of the
loop agree by construction.


In [0]:
feature_pipeline = PipelineModel.load(f"{config.ARTIFACT_VOLUME}/feature_pipeline")
api_silver = spark.table(config.API_SILVER)
print(f"Rows to score: {api_silver.count():,}")

manifest = spark.table(config.FEATURE_MANIFEST).orderBy("vector_index").collect()
index_by_name = {r["name"]: int(r["vector_index"]) for r in manifest}
n_features = len(index_by_name)

dep_delay_idx = index_by_name["dep_delay"]
pre_indices = [i for i in range(n_features) if i != dep_delay_idx]
assert dep_delay_idx not in pre_indices
print(f"Vector width {n_features}; dep_delay at {dep_delay_idx} (from the manifest)")


Rows to score: 162
Vector width 816; dep_delay at 11 (from the manifest)


In [0]:
# Fill only the columns the pipeline actually consumes, matching 04_gold's treatment.
numeric_like = [f.name for f in api_silver.schema.fields
                if f.dataType.typeName() in ("double", "integer", "long", "float")]
string_like = [f.name for f in api_silver.schema.fields if f.dataType.typeName() == "string"]

api_prepared = (
    api_silver
    .withColumn("dep_hour", (F.col("crs_dep_time") / 100).cast("int"))
    .withColumn("arr_hour", (F.col("crs_arr_time") / 100).cast("int"))
    .na.fill(0, subset=numeric_like)
    .na.fill("UNKNOWN", subset=string_like)
)

# Scoring-time feature parity.
#
# 04_gold's pipeline now expects the congestion features 03_silver builds, and
# api_silver_flights has none of them — this is what FIELD_NOT_FOUND on
# `sched_deps_origin_hour` was. Whatever the model was trained on has to exist
# here under the same names, or the transform cannot run.
#
# The two the model actually uses are reconstructed below. They are handled
# differently because they fail differently on a partial feed:
#
#   schedule_padding      needs a route median, which is a stable property of the
#                         schedule. Taken from Silver, where it is computed over
#                         four years rather than over whatever the API returned.
#
#   dep_sequence_in_day   is an ordinal within a carrier's departures from an
#                         airport that day. The API returns one route, not a full
#                         airport-day, so the sequence is computed over the flights
#                         actually fetched and is an under-count of the true
#                         position. Documented in docs/API_STRATEGY.md: the fix is
#                         to pull schedules per airport-day, which is the ingestion
#                         shape the congestion features require anyway.
route_medians = (
    spark.table(config.SILVER)
    .groupBy("origin_airport_code", "destination_airport_code")
    .agg(F.expr("percentile_approx(crs_elapsed_time, 0.5)").alias("route_median_elapsed"))
)
global_median = (
    spark.table(config.SILVER)
    .agg(F.expr("percentile_approx(crs_elapsed_time, 0.5)")).first()[0]
)

carrier_day = (
    Window.partitionBy("origin_airport_code", "airline_code", "flight_date")
    .orderBy("dep_minutes")
)

# Rebuild the congestion features that 03_silver creates and the pipeline expects.
origin_hour_win = Window.partitionBy(
    "origin_airport_code", "flight_date", "dep_hour"
)
bank_win = (
    Window.partitionBy("origin_airport_code", "flight_date")
    .orderBy("dep_minutes")
    .rangeBetween(-60, 60)
)

api_prepared = (
    api_prepared
    .withColumn("dep_minutes",
                (F.floor(F.col("crs_dep_time") / 100) * 60
                 + (F.col("crs_dep_time") % 100)).cast("int"))
    .join(F.broadcast(route_medians),
          on=["origin_airport_code", "destination_airport_code"], how="left")
    .withColumn("route_median_elapsed",
                F.coalesce(F.col("route_median_elapsed"), F.lit(global_median)))
    .withColumn("schedule_padding",
                F.col("crs_elapsed_time") - F.col("route_median_elapsed"))
    .withColumn("dep_sequence_in_day", F.row_number().over(carrier_day))
    .withColumn("sched_deps_origin_hour", F.count("*").over(origin_hour_win))
    .withColumn("dep_bank_density", F.count("*").over(bank_win))
    .drop("route_median_elapsed")
)

unmatched = api_prepared.filter(F.col("schedule_padding").isNull()).count()
print(f"Scoring-time features rebuilt. Rows with no route median: {unmatched}")
print(f"  (those fall back to the global median of {global_median:.0f} min)")

# Fail here, with the missing names, rather than inside the pipeline transform.
expected = set(spark.table(config.FEATURE_MANIFEST).select("source_column")
               .distinct().toPandas()["source_column"])
base_cols = {c for c in expected if not c.endswith("_ohe")}
missing = sorted(base_cols - set(api_prepared.columns))
if missing:
    raise ValueError(
        f"api_silver_flights is missing {missing}, which 04_gold's pipeline expects. "
        "Either 03_silver added features that scoring does not rebuild, or the feature "
        "pipeline is newer than the API projection in 06_api_ingest."
    )
print(f"All {len(base_cols)} pipeline input columns present.")

transformed = feature_pipeline.transform(api_prepared)
scored_input = VectorSlicer(
    inputCol="features", outputCol="features_pre", indices=pre_indices
).transform(transformed)


Scoring-time features rebuilt. Rows with no route median: 0
  (those fall back to the global median of 126 min)
All 18 pipeline input columns present.


## Score both variants

Each champion is a `PipelineModel` containing its feature selector *and* its classifier,
and is applied whole. The previous version pulled `model.stages[0]` out and called it "the
classifier" — which after the `05_train` rewrite is the selector, and which in any case
applied a different feature space at scoring than the model was trained on. Logging the
selector and classifier as one artifact only helps if scoring uses the artifact.

Because both models expect their features in a column called `features`, the pre-departure
view is renamed into place rather than the model being reconfigured.


In [0]:
def score_variant(df, model, features_col, threshold, prefix):
    """Apply a champion pipeline whole and emit probability + decision at its threshold."""
    renamed = df.withColumn("_orig_features", F.col("features")).drop("features") \
                .withColumnRenamed(features_col, "features")

    out = model.transform(renamed)
    out = (
        out.withColumn(f"prob_{prefix}", vector_to_array("probability")[1])
           .withColumn(f"pred_{prefix}",
                       (F.col(f"prob_{prefix}") >= F.lit(threshold)).cast("int"))
           .drop("rawPrediction", "probability", "prediction", "selected", "features")
           .withColumnRenamed("_orig_features", "features")
    )
    return out


scored = score_variant(scored_input, pre_model, "features_pre", PRE_THRESHOLD, "pre")
scored = score_variant(
    scored.withColumn("features_all", F.col("features")),
    in_model, "features_all", IN_THRESHOLD, "in",
)

# Risk bands are anchored on each model's own threshold rather than on 0.5/0.7.
# The pre-departure cut is well below 0.5, so fixed bands put every flight in "Low"
# and the table would say nothing.
def risk_band(prob_col, threshold):
    return (
        F.when(F.col(prob_col) >= threshold * 1.5, F.lit("High"))
         .when(F.col(prob_col) >= threshold, F.lit("Medium"))
         .otherwise(F.lit("Low"))
    )


scored = (
    scored
    .withColumn("risk_pre", risk_band("prob_pre", PRE_THRESHOLD))
    .withColumn("risk_in", risk_band("prob_in", IN_THRESHOLD))
)
print(f"Bands — pre-departure: Medium >= {PRE_THRESHOLD:.2f}, High >= {PRE_THRESHOLD * 1.5:.2f}")
print(f"        in-flight    : Medium >= {IN_THRESHOLD:.2f}, High >= {IN_THRESHOLD * 1.5:.2f}")


Bands — pre-departure: Medium >= 0.16, High >= 0.24
        in-flight    : Medium >= 0.39, High >= 0.58


### Routing each flight to the model that fits its phase

`06_api_ingest` attaches `flight_phase` from OpenSky's `on_ground` flag. Both models are still
applied to every row — scoring is cheap and the comparison is informative — but
`recommended_model` records which one is *valid* for each flight, and the summary reports the
split.

This matters because the in-flight model is only meaningful once `dep_delay` exists. Serving
its output for a flight that has not left the gate means serving a prediction built on a
feature whose value is not yet known, which is a leak at inference time rather than at
training time.


In [0]:
# Phase comes from OpenSky via 06_api_ingest. Older API Silver tables predate the
# column, so fall back rather than fail.
if "flight_phase" not in scored.columns:
    scored = scored.withColumn("flight_phase", F.lit("unknown"))
    print("No flight_phase column — re-run 06_api_ingest with USE_OPENSKY=true.")

scored = scored.withColumn(
    "recommended_model",
    F.when(
        (F.col("flight_phase") == "airborne") & F.col("dep_delay").isNotNull(),
        F.lit("in_flight"),
    ).otherwise(F.lit("pre_departure")),
).withColumn(
    "recommended_prob_pct",
    F.when(F.col("recommended_model") == "in_flight", F.col("prob_in") * 100)
     .otherwise(F.col("prob_pre") * 100),
)

display(
    scored.groupBy("flight_phase", "recommended_model")
    .agg(F.count("*").alias("flights"),
         F.round(F.avg("recommended_prob_pct"), 2).alias("avg_delay_prob_pct"))
    .orderBy("flight_phase")
)
print("`unknown` falls back to pre-departure: it is the variant that does not")
print("require a departure to have already happened, so it is the safe default.")


flight_phase,recommended_model,flights,avg_delay_prob_pct
UNKNOWN,pre_departure,62,21.46
unknown,pre_departure,100,22.56


`unknown` falls back to pre-departure: it is the variant that does not
require a departure to have already happened, so it is the safe default.


## Predictions — MERGE, not overwrite

The previous version wrote `mode("overwrite")`, so every run destroyed the last run's
predictions. For a table that is supposed to represent live scoring that is the wrong
semantics twice over: there is no record of what was predicted before the flight departed,
which is exactly the record you need to evaluate the model later.

A `MERGE` on (flight, date, scoring run) makes the job **idempotent** — re-running after a
failure updates in place instead of duplicating — and keeps history across runs. This is
the pattern the medallion architecture exists to enable, and it is the one thing this
pipeline was not using Delta for.


In [0]:
# Scoring-time schema guard: a clear failure here beats a lazy one downstream.
try:
    scored_cols = scored.columns
    if not scored_cols:
        raise RuntimeError("scored DataFrame resolved to no columns")
except Exception as e:
    raise RuntimeError(
        "Upstream failure in the feature-pipeline transform.\n\n"
        "04_gold's pipeline expects columns api_silver does not have. Re-run "
        "04_gold (refit_pipeline=auto now refits when the feature set changes), "
        "then 05_train.\n\n"
        f"Original error: {e}"
    ) from e


# ---------------------------------------------------------------------------
# The verdict
# ---------------------------------------------------------------------------
# Everything below exists because a probability is not an answer. The model has
# already made a decision — probability against its own tuned threshold — and a
# table that reports 0.34 and a threshold of 0.17 in separate columns is asking
# the reader to re-derive that decision on every row.
#
# `recommended_model` decides which variant is valid for each flight: the
# in-flight model only means something once dep_delay exists.
ACTIVE_PROB = F.when(F.col("recommended_model") == "in_flight", F.col("prob_in")) \
               .otherwise(F.col("prob_pre"))
ACTIVE_THRESHOLD = F.when(F.col("recommended_model") == "in_flight", F.lit(IN_THRESHOLD)) \
                    .otherwise(F.lit(PRE_THRESHOLD))

scored_v = (
    scored
    .withColumn("active_prob", ACTIVE_PROB)
    .withColumn("active_threshold", ACTIVE_THRESHOLD)
    .withColumn("will_be_delayed", (F.col("active_prob") >= F.col("active_threshold")).cast("int"))
    # Readable identity. "DL1234" is what a person calls this flight; airline_code
    # and fl_number in adjacent columns are what a database calls it.
    .withColumn("flight", F.concat(F.col("airline_code"), F.col("fl_number").cast("string")))
    .withColumn(
        "scheduled_departure",
        F.concat_ws(
            ":",
            F.lpad(F.floor(F.col("crs_dep_time") / 100).cast("int").cast("string"), 2, "0"),
            F.lpad((F.col("crs_dep_time") % 100).cast("int").cast("string"), 2, "0"),
        ),
    )
    .withColumn(
        "scheduled_arrival",
        F.concat_ws(
            ":",
            F.lpad(F.floor(F.col("crs_arr_time") / 100).cast("int").cast("string"), 2, "0"),
            F.lpad((F.col("crs_arr_time") % 100).cast("int").cast("string"), 2, "0"),
        ),
    )
    .withColumn(
        "prediction",
        F.when(F.col("will_be_delayed") == 1, F.lit("DELAY EXPECTED"))
         .otherwise(F.lit("ON TIME")),
    )
    # Distance from the threshold, relative to the room available on that side.
    # A flight at 0.18 against a 0.17 cut is a coin flip; one at 0.80 is not, and
    # the table should not present them identically.
    .withColumn(
        "margin",
        F.when(
            F.col("will_be_delayed") == 1,
            (F.col("active_prob") - F.col("active_threshold"))
            / F.greatest(F.lit(1.0) - F.col("active_threshold"), F.lit(0.01)),
        ).otherwise(
            (F.col("active_threshold") - F.col("active_prob"))
            / F.greatest(F.col("active_threshold"), F.lit(0.01))
        ),
    )
    .withColumn(
        "confidence",
        F.when(F.col("margin") >= 0.60, F.lit("High"))
         .when(F.col("margin") >= 0.25, F.lit("Moderate"))
         .otherwise(F.lit("Marginal")),
    )
    .withColumn(
        "basis",
        F.when(F.col("recommended_model") == "in_flight",
               F.lit("in-flight (departure delay known)"))
         .otherwise(F.lit("pre-departure (schedule only)")),
    )
    .withColumn(
        "explanation",
        F.concat(
            F.col("flight"), F.lit(" "),
            F.col("origin_airport_code"), F.lit("-"), F.col("destination_airport_code"),
            F.lit(" departing "), F.col("scheduled_departure"),
            F.lit(" on "), F.col("flight_date").cast("string"),
            F.lit(": "), F.col("prediction"),
            F.lit(" ("), F.round(F.col("active_prob") * 100, 1).cast("string"),
            F.lit("% chance of arriving 15+ min late, threshold "),
            F.round(F.col("active_threshold") * 100, 0).cast("int").cast("string"),
            F.lit("%, "), F.col("confidence"), F.lit(" confidence)"),
        ),
    )
)

predictions = scored_v.select(
    F.current_timestamp().alias("prediction_timestamp"),
    F.current_date().alias("scoring_date"),
    # --- what a person reads -------------------------------------------------
    "flight", "airline_name",
    F.concat_ws(" -> ", "origin_airport_code", "destination_airport_code").alias("route"),
    "flight_date", "scheduled_departure", "scheduled_arrival",
    "prediction", "confidence", "basis",
    (F.col("active_prob") * 100).alias("delay_probability_pct"),
    "explanation",
    # --- what a system reads -------------------------------------------------
    "airline_code", "fl_number", "origin_airport_code", "destination_airport_code",
    "crs_dep_time", "crs_arr_time", "dep_delay",
    "will_be_delayed", "flight_phase", "recommended_model",
    (F.col("active_threshold")).alias("applied_threshold"),
    (F.col("prob_pre") * 100).alias("prob_delay_pre_pct"),
    F.col("pred_pre").alias("predicted_delayed_pre"),
    F.col("risk_pre").alias("risk_pre_departure"),
    (F.col("prob_in") * 100).alias("prob_delay_in_pct"),
    F.col("pred_in").alias("predicted_delayed_in"),
    F.col("risk_in").alias("risk_in_flight"),
    F.lit(pre_name).alias("model_pre"),
    F.lit(str(pre_version)).alias("model_pre_version"),
    F.lit(PRE_THRESHOLD).alias("threshold_pre"),
    F.lit(in_name).alias("model_in"),
    F.lit(str(in_version)).alias("model_in_version"),
    F.lit(IN_THRESHOLD).alias("threshold_in"),
)

# Explicit types. The previous version guessed from column-name prefixes, which
# silently mistypes anything that does not follow the convention.
COLUMN_TYPES = {
    "prediction_timestamp": "TIMESTAMP", "scoring_date": "DATE",
    "flight": "STRING", "airline_name": "STRING", "route": "STRING",
    "flight_date": "DATE", "scheduled_departure": "STRING",
    "scheduled_arrival": "STRING", "prediction": "STRING",
    "confidence": "STRING", "basis": "STRING",
    "delay_probability_pct": "DOUBLE", "explanation": "STRING",
    "airline_code": "STRING", "fl_number": "INT",
    "origin_airport_code": "STRING", "destination_airport_code": "STRING",
    "crs_dep_time": "INT", "crs_arr_time": "INT", "dep_delay": "DOUBLE",
    "will_be_delayed": "INT", "flight_phase": "STRING",
    "recommended_model": "STRING", "applied_threshold": "DOUBLE",
    "prob_delay_pre_pct": "DOUBLE", "predicted_delayed_pre": "INT",
    "risk_pre_departure": "STRING", "prob_delay_in_pct": "DOUBLE",
    "predicted_delayed_in": "INT", "risk_in_flight": "STRING",
    "model_pre": "STRING", "model_pre_version": "STRING", "threshold_pre": "DOUBLE",
    "model_in": "STRING", "model_in_version": "STRING", "threshold_in": "DOUBLE",
}
expected_cols = list(COLUMN_TYPES)

if spark.catalog.tableExists(config.PREDICTIONS):
    missing = [c for c in expected_cols
               if c not in set(spark.table(config.PREDICTIONS).columns)]
    if missing:
        cols_ddl = ", ".join(f"{c} {COLUMN_TYPES[c]}" for c in missing)
        spark.sql(f"ALTER TABLE {config.PREDICTIONS} ADD COLUMNS ({cols_ddl})")
        print(f"Added columns to {config.PREDICTIONS}: {missing}")

# One row per flight per scoring run: MERGE rejects a source that matches a
# target row more than once, and api_silver accumulates across ingestion runs.
dedup_window = Window.partitionBy(
    "airline_code", "fl_number", "flight_date", "scoring_date"
).orderBy(F.desc("prediction_timestamp"))

predictions_deduped = (
    predictions
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

dup_count = predictions.count() - predictions_deduped.count()
if dup_count:
    print(f"Collapsed {dup_count:,} duplicate rows before MERGE "
          f"(api_silver accumulates across ingestion runs)")

if not spark.catalog.tableExists(config.PREDICTIONS):
    (predictions_deduped.limit(0).write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(config.PREDICTIONS))
    print(f"Created {config.PREDICTIONS}")

(
    DeltaTable.forName(spark, config.PREDICTIONS).alias("t")
    .merge(
        predictions_deduped.alias("s"),
        "t.airline_code = s.airline_code AND t.fl_number = s.fl_number "
        "AND t.flight_date = s.flight_date AND t.scoring_date = s.scoring_date",
    )
    .whenMatchedUpdate(set={c: F.col(f"s.{c}") for c in expected_cols})
    .whenNotMatchedInsert(values={c: F.col(f"s.{c}") for c in expected_cols})
    .execute()
)

total = spark.table(config.PREDICTIONS).count()
dates = spark.table(config.PREDICTIONS).select("scoring_date").distinct().count()
print(f"MERGE complete. {config.PREDICTIONS}: {total:,} rows across {dates} scoring date(s).")


Added columns to workspace.flights.flight_delay_predictions: ['flight', 'scheduled_departure', 'scheduled_arrival', 'prediction', 'confidence', 'basis', 'delay_probability_pct', 'explanation', 'will_be_delayed', 'flight_phase', 'recommended_model', 'applied_threshold']
Collapsed 8 duplicate rows before MERGE (api_silver accumulates across ingestion runs)
MERGE complete. workspace.flights.flight_delay_predictions: 324 rows across 4 scoring date(s).


## The departure board

What the model actually predicts, for which flight, in words.

The table above carries `delay_probability_pct` and `applied_threshold` as separate columns
because a downstream system needs both. A person does not: they need to know whether this
flight is expected to be late. The model already made that call — probability against its own
tuned threshold — so the verdict is written down rather than left for the reader to re-derive
on every row.

**`confidence` is distance from the threshold, not probability.** A pre-departure flight at
18% against a 17% cut is a coin flip; one at 80% is not, and a table that shows both as
"delayed" without distinguishing them is hiding the more useful number. Margin is scaled by
the room available on each side of the cut, because the threshold is not 0.5 and the two
sides are not symmetric.

**`basis` says which model answered.** The in-flight model only means anything once departure
delay is known; before pushback it would be predicting from a feature whose value does not
exist yet.


In [0]:
board = (
    spark.table(config.PREDICTIONS)
    .filter(F.col("scoring_date") == F.current_date())
    .orderBy(F.desc("delay_probability_pct"))
)
n = board.count()

if n == 0:
    print("No predictions for today. Run 06_api_ingest first.")
else:
    display(
        board.select(
            F.col("flight").alias("FLIGHT"),
            F.col("route").alias("ROUTE"),
            F.col("flight_date").alias("DATE"),
            F.col("scheduled_departure").alias("DEP"),
            F.col("prediction").alias("PREDICTION"),
            F.round(F.col("delay_probability_pct"), 1).alias("P(DELAY) %"),
            F.col("confidence").alias("CONFIDENCE"),
            F.col("basis").alias("BASIS"),
        )
    )

    summary = board.agg(
        F.count("*").alias("flights"),
        F.sum("will_be_delayed").alias("flagged"),
        F.avg("delay_probability_pct").alias("avg_pct"),
    ).first()
    flagged = summary["flagged"] or 0

    print("=" * 78)
    print(f"{n} flights scored on {spark.sql('SELECT current_date()').first()[0]}")
    print("=" * 78)
    print(f"  DELAY EXPECTED : {flagged:>4}  ({flagged / n:.0%})")
    print(f"  ON TIME        : {n - flagged:>4}  ({(n - flagged) / n:.0%})")
    print(f"  mean P(delay)  : {summary['avg_pct']:.1f}%")

    by_conf = (
        board.groupBy("prediction", "confidence")
        .agg(F.count("*").alias("n"))
        .orderBy("prediction", "confidence")
        .toPandas()
    )
    print("\n  breakdown by confidence:")
    for _, r in by_conf.iterrows():
        print(f"    {r['prediction']:<16}{r['confidence']:<10}{int(r['n']):>5}")

    print("\n" + "-" * 78)
    print("Highest-risk flights, as sentences:")
    print("-" * 78)
    for r in board.select("explanation").limit(5).collect():
        print(f"  {r['explanation']}")

    print("\n" + "-" * 78)
    print("Lowest-risk flights:")
    print("-" * 78)
    for r in board.orderBy("delay_probability_pct").select("explanation").limit(3).collect():
        print(f"  {r['explanation']}")

    print("\nEvery verdict compares this flight's probability against the threshold")
    print("its own model was tuned to, read off the model version at load time.")
    print("Nothing here uses 0.5, which for the pre-departure model predicts zero")
    print("delays and scores F1 = 0.0000.")


FLIGHT,ROUTE,DATE,DEP,PREDICTION,P(DELAY) %,CONFIDENCE,BASIS
DL753,ATL -> LAX,2026-08-31,18:47,DELAY EXPECTED,25.2,Marginal,pre-departure (schedule only)
Y42547,ATL -> LAX,2026-08-31,19:15,DELAY EXPECTED,24.6,Marginal,pre-departure (schedule only)
DL895,ATL -> LAX,2026-08-31,16:35,DELAY EXPECTED,24.6,Marginal,pre-departure (schedule only)
AM4626,ATL -> LAX,2026-08-31,18:47,DELAY EXPECTED,24.5,Marginal,pre-departure (schedule only)
WS6993,ATL -> LAX,2026-08-31,18:47,DELAY EXPECTED,24.5,Marginal,pre-departure (schedule only)
VS5147,ATL -> LAX,2026-08-31,18:47,DELAY EXPECTED,24.4,Marginal,pre-departure (schedule only)
F93215,ATL -> LAX,2026-08-31,19:15,DELAY EXPECTED,24.4,Marginal,pre-departure (schedule only)
AF5859,ATL -> LAX,2026-08-31,18:47,DELAY EXPECTED,24.4,Marginal,pre-departure (schedule only)
LA6425,ATL -> LAX,2026-08-31,16:35,DELAY EXPECTED,24.4,Marginal,pre-departure (schedule only)
WS7034,ATL -> LAX,2026-08-31,16:35,DELAY EXPECTED,24.4,Marginal,pre-departure (schedule only)


154 flights scored on 2026-09-14
  DELAY EXPECTED :  149  (97%)
  ON TIME        :    5  (3%)
  mean P(delay)  : 22.4%

  breakdown by confidence:
    DELAY EXPECTED  Marginal    149
    ON TIME         Marginal      5

------------------------------------------------------------------------------
Highest-risk flights, as sentences:
------------------------------------------------------------------------------
  DL753 ATL-LAX departing 18:47 on 2026-08-31: DELAY EXPECTED (25.2% chance of arriving 15+ min late, threshold 16%, Marginal confidence)
  Y42547 ATL-LAX departing 19:15 on 2026-08-31: DELAY EXPECTED (24.6% chance of arriving 15+ min late, threshold 16%, Marginal confidence)
  DL895 ATL-LAX departing 16:35 on 2026-08-31: DELAY EXPECTED (24.6% chance of arriving 15+ min late, threshold 16%, Marginal confidence)
  WS6993 ATL-LAX departing 18:47 on 2026-08-31: DELAY EXPECTED (24.5% chance of arriving 15+ min late, threshold 16%, Marginal confidence)
  AM4626 ATL-LAX departing 18:47

## Alternative-flight recommender

Same origin and destination, same day, within ±3 hours, at least 10 points lower delay
probability. Ranked by improvement plus a bonus for landing in a lower risk band, top 5
per flight.

**On the time window.** The previous version compared raw `HHMM` integers and called a
difference of 300 "three hours". `HHMM` is not linear in time — 13:00 minus 12:59 is 41 in
that arithmetic, and one minute in reality. Departure times are converted to minutes since
midnight first, which is what makes the window mean what it says.


In [0]:
def hhmm_to_minutes(c):
    return (F.floor(c / 100) * 60 + (c % 100)).cast("int")


scored_pool = (
    spark.table(config.PREDICTIONS)
    .filter(F.col("scoring_date") == F.current_date())
    .withColumn("dep_minutes", hhmm_to_minutes(F.col("crs_dep_time")))
)

candidates = scored_pool.selectExpr(
    "airline_name AS alt_airline", "airline_code AS alt_airline_code",
    "fl_number AS alt_flight", "origin_airport_code", "destination_airport_code",
    "flight_date", "crs_dep_time AS alt_crs_dep_time", "dep_minutes AS alt_dep_minutes",
    "delay_probability_pct AS alt_prob_delay_pct", "prediction AS alt_prediction",
    "confidence AS alt_confidence", "dep_delay AS alt_dep_delay",
)

WINDOW_MINUTES = 180
MIN_IMPROVEMENT_PCT = 10.0

same_route = (
    scored_pool.alias("orig")
    .join(
        candidates.alias("alt"),
        (F.col("orig.origin_airport_code") == F.col("alt.origin_airport_code"))
        & (F.col("orig.destination_airport_code") == F.col("alt.destination_airport_code"))
        & (F.col("orig.flight_date") == F.col("alt.flight_date"))
        & (F.col("orig.fl_number") != F.col("alt.alt_flight")),
    )
)
in_window = same_route.filter(
    F.abs(F.col("orig.dep_minutes") - F.col("alt.alt_dep_minutes")) <= WINDOW_MINUTES
)
improved = (
    in_window
    .withColumn("improvement_pct",
                F.col("orig.delay_probability_pct") - F.col("alt.alt_prob_delay_pct"))
    .filter(F.col("improvement_pct") >= MIN_IMPROVEMENT_PCT)
)

# Each filter's survivor count, so an empty result names its own cause. "0
# recommendations" is equally consistent with a bug, an empty input, and a
# correct answer on a single-route feed — and those need different responses.
n_pool, n_pairs = scored_pool.count(), same_route.count()
n_window, n_improved = in_window.count(), improved.count()

print(f"{'flights scored today':<46}{n_pool:>10,}")
print(f"{'same-route, same-day pairs':<46}{n_pairs:>10,}")
print(f"{f'within +/-{WINDOW_MINUTES} min':<46}{n_window:>10,}")
print(f"{f'at least {MIN_IMPROVEMENT_PCT:.0f}pp better':<46}{n_improved:>10,}")

if n_improved == 0:
    print("\nNo recommendations, and the funnel says why:")
    if n_pool < 2:
        print("  Fewer than two flights scored — nothing to compare against.")
    elif n_pairs == 0:
        print("  No two flights share a route and date. 06_api_ingest fetches one")
        print("  origin/destination pair at a time, so alternatives only exist once")
        print("  ingestion pulls whole airport-days (docs/API_STRATEGY.md).")
    elif n_window == 0:
        print(f"  Pairs exist but none depart within {WINDOW_MINUTES} minutes of each other.")
    else:
        print(f"  {n_window:,} pairs are close enough in time, but none differ by")
        print(f"  {MIN_IMPROVEMENT_PCT:.0f} percentage points. Expected on a single route:")
        print("  flights sharing an origin, destination and hour see nearly identical")
        print("  features, so the model scores them nearly identically. The recommender")
        print("  needs variety in the candidate pool, which is an ingestion property,")
        print("  not a modelling one.")

recommendations = (
    improved
    .withColumn("risk_bonus",
                F.when(F.col("alt.alt_prediction") == "ON TIME", 20)
                 .when(F.col("alt.alt_confidence") == "High", 10).otherwise(0))
    .withColumn("recommendation_score", F.col("improvement_pct") + F.col("risk_bonus"))
    .withColumn(
        "recommendation_rank",
        F.row_number().over(
            Window.partitionBy("orig.airline_code", "orig.fl_number", "orig.flight_date")
            .orderBy(F.col("recommendation_score").desc())
        ),
    )
    .filter(F.col("recommendation_rank") <= 5)
    .select(
        F.col("orig.flight").alias("original_flight"),
        F.col("orig.airline_name").alias("original_airline"),
        F.col("orig.origin_airport_code").alias("origin"),
        F.col("orig.destination_airport_code").alias("destination"),
        F.col("orig.flight_date").alias("flight_date"),
        F.col("orig.scheduled_departure").alias("original_departure"),
        F.col("orig.prediction").alias("original_prediction"),
        F.col("orig.delay_probability_pct").alias("original_delay_prob"),
        F.col("alt.alt_airline").alias("alternative_airline"),
        F.col("alt.alt_flight").alias("alternative_flight"),
        F.col("alt.alt_prediction").alias("alternative_prediction"),
        F.col("alt.alt_prob_delay_pct").alias("alternative_delay_prob"),
        F.col("alt.alt_confidence").alias("alternative_confidence"),
        "improvement_pct", "recommendation_score", "recommendation_rank",
        F.current_timestamp().alias("recommendation_timestamp"),
    )
)

(
    recommendations.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(config.ALTERNATIVES)
)
written = recommendations.count()
print(f"\nWrote {written:,} recommendations -> {config.ALTERNATIVES}")

if written:
    display(recommendations.orderBy(F.desc("recommendation_score")).limit(10))
    for r in recommendations.orderBy(F.desc("improvement_pct")).limit(3).collect():
        print(f"  {r['original_flight']} ({r['original_delay_prob']:.0f}% delay risk) "
              f"-> {r['alternative_airline']} {r['alternative_flight']} "
              f"({r['alternative_delay_prob']:.0f}%), "
              f"{r['improvement_pct']:.0f}pp better")


flights scored today                                 154
same-route, same-day pairs                         2,772
within +/-180 min                                  1,284
at least 10pp better                                   0

No recommendations, and the funnel says why:
  1,284 pairs are close enough in time, but none differ by
  10 percentage points. Expected on a single route:
  flights sharing an origin, destination and hour see nearly identical
  features, so the model scores them nearly identically. The recommender
  needs variety in the candidate pool, which is an ingestion property,
  not a modelling one.

Wrote 0 recommendations -> workspace.flights.alternative_flight_recommendations


## Loop closed

Every model above was loaded by `models:/catalog.schema.name@champion` — no run ID, no
version pinned in code — and scored at a threshold read off the model itself. Retraining
promotes a new champion and this notebook picks it up unchanged.

That is the defect the original project died on, demonstrated working end to end.


In [0]:
summary = spark.sql(f"""
    SELECT scoring_date,
           COUNT(*)                                   AS flights_scored,
           SUM(predicted_delayed_pre)                 AS flagged_pre_departure,
           SUM(predicted_delayed_in)                  AS flagged_in_flight,
           ROUND(AVG(prob_delay_pre_pct), 2)          AS avg_pre_pct,
           ROUND(AVG(prob_delay_in_pct), 2)           AS avg_in_pct
    FROM {config.PREDICTIONS}
    GROUP BY scoring_date ORDER BY scoring_date DESC
""")
display(summary)

print(f"pre-departure champion : {pre_name} v{pre_version} @ {PRE_THRESHOLD:.2f}")
print(f"in-flight champion     : {in_name} v{in_version} @ {IN_THRESHOLD:.2f}")


scoring_date,flights_scored,flagged_pre_departure,flagged_in_flight,avg_pre_pct,avg_in_pct
2026-09-14,154,149,0,22.4,5.2
2026-09-13,60,55,0,20.05,6.23
2026-09-10,56,46,0,18.91,7.57
null,54,null,null,22.49,13.25


pre-departure champion : workspace.flights.rf_pre_departure v8 @ 0.16
in-flight champion     : workspace.flights.rf_in_flight v6 @ 0.39
